In [ ]:
import os
import torch
import pandas as pd
import librosa
import numpy as np
import json
from torch.utils.data import Dataset
from transformers import (
    SpeechT5Processor, 
    SpeechT5ForTextToSpeech, 
    Seq2SeqTrainingArguments, 
    Seq2SeqTrainer,
    SpeechT5FeatureExtractor,
    BertTokenizer
)
from dataclasses import dataclass
from typing import Any, Dict, List

# 1. PATHS & CONFIG

LOCAL_MODEL_PATH = r"C:/TTS-Speech_t5/speecht5-sin"
DATASET_DIR = r"C:/TTS-Speech_t5/sinhala_vits" 
OUTPUT_DIR = "./sinhala_tts_output"

device = "cuda" if torch.cuda.is_available() else "cpu"

# 2. FIX MISSING PREPROCESSOR

prepro_path = os.path.join(LOCAL_MODEL_PATH, "preprocessor_config.json")
if not os.path.exists(prepro_path):
    config = {
        "feature_extractor_type": "SpeechT5FeatureExtractor",
        "feature_size": 1, "num_mel_bins": 80, "hop_length": 256,
        "win_length": 1024, "sampling_rate": 16000, "normalize_speech": True,
        "do_normalize": True, "return_attention_mask": True
    }
    with open(prepro_path, "w") as f:
        json.dump(config, f)
    print("Created missing preprocessor_config.json")

# 3. LOAD MODEL & PROCESSOR

feature_extractor = SpeechT5FeatureExtractor.from_pretrained(LOCAL_MODEL_PATH)
tokenizer = BertTokenizer.from_pretrained(LOCAL_MODEL_PATH)
processor = SpeechT5Processor(feature_extractor=feature_extractor, tokenizer=tokenizer)
model = SpeechT5ForTextToSpeech.from_pretrained(LOCAL_MODEL_PATH).to(device)

# 4. CUSTOM DATASET CLASS

class LocalTTSDataset(Dataset):
    def __init__(self, csv_file, base_dir, processor):
        self.df = pd.read_csv(csv_file)
        self.base_dir = base_dir
        self.processor = processor

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        audio_path = os.path.join(self.base_dir, row["file_name"])
        
        speech, _ = librosa.load(audio_path, sr=16000)
        
        inputs = self.processor(
            text=row["text"],
            audio_target=speech,
            sampling_rate=16000,
            return_attention_mask=False,
        )

        return {
            "input_ids": inputs["input_ids"],
            "labels": torch.tensor(inputs["labels"][0]),
            "speaker_embeddings": torch.zeros(512) 
        }

train_ds = LocalTTSDataset(os.path.join(DATASET_DIR, "metadata_train.csv"), DATASET_DIR, processor)
test_ds = LocalTTSDataset(os.path.join(DATASET_DIR, "metadata_test.csv"), DATASET_DIR, processor)

# 5. DATA COLLATOR (FIXED)

@dataclass
class TTSDataCollatorWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
        input_ids = [{"input_ids": feature["input_ids"]} for feature in features]
        label_features = [{"input_values": feature["labels"]} for feature in features]
        speaker_features = [feature["speaker_embeddings"] for feature in features]

        # Pad text
        batch = self.processor.pad(input_ids=input_ids, return_tensors="pt")
        
        # Pad spectrograms
        labels_batch = self.processor.pad(labels=label_features, return_tensors="pt")
        
        # Step 1: Extract the padded values
        labels = labels_batch["input_values"]

        # Step 2: Ensure even number of frames for SpeechT5 decoder
        if labels.shape[1] % 2 != 0:
            labels = labels[:, :-1, :]

        # Step 3: Assign to batch
        batch["labels"] = labels
        batch["speaker_embeddings"] = torch.stack(speaker_features)

        return batch

collator = TTSDataCollatorWithPadding(processor=processor)

# 6. TRAINING & EXECUTION

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,
    learning_rate=5e-6,
    warmup_steps=100,
    max_steps=10000, 
    save_steps=500,
    eval_strategy="steps",
    fp16=True if torch.cuda.is_available() else False,
    logging_steps=10,
    save_total_limit=2,
    remove_unused_columns=False,
    label_names=["labels"],
    full_determinism=True
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    processing_class=processor.tokenizer,
    data_collator=collator,
)

# --- Final Check ---
print("Checking Collator Output...")
sample_batch = [train_ds[0], train_ds[1]]
mock_batch = collator(sample_batch)
print(f"Collator Keys: {mock_batch.keys()}") 
print(f"Labels Shape: {mock_batch['labels'].shape}") 

print("Starting training loop...")
trainer.train()

# Save final model
final_path = os.path.join(OUTPUT_DIR, "final_sinhala_speecht5")
trainer.save_model(final_path)
processor.save_pretrained(final_path)
print(f"Success! Model saved to {final_path}")

Loading weights: 100%|██████████| 396/396 [00:00<00:00, 552.00it/s, Materializing param=speecht5.encoder.wrapped_encoder.layers.11.layer_norm.weight]                     
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 68, 'bos_token_id': 67, 'pad_token_id': 69}.


Checking Collator Output...
Collator Keys: KeysView({'input_ids': tensor([[72,  0,  0,  0,  0,  0,  0, 32,  0,  0,  0,  0,  0,  0,  0,  0, 71],
        [72,  0,  0,  0,  0,  0,  0,  0, 71, 69, 69, 69, 69, 69, 69, 69, 69]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0]]), 'labels': tensor([[[-1.2377, -1.6297, -1.2344,  ..., -2.5835, -2.5615, -2.5196],
         [-1.3667, -1.7346, -1.5028,  ..., -2.5769, -2.6233, -2.6587],
         [-1.6626, -1.8950, -2.1692,  ..., -2.4224, -2.5031, -2.6383],
         ...,
         [-0.5778, -0.1247, -0.3635,  ..., -2.3359, -2.3962, -2.3425],
         [-0.3461, -0.1884, -0.5965,  ..., -2.4968, -2.5543, -2.6273],
         [-0.5576, -0.4857, -0.7180,  ..., -2.8002, -2.7104, -2.6985]],

        [[-2.8712, -2.8016, -3.0890,  ..., -3.5922, -3.8322, -3.7817],
         [-1.8754, -1.4465, -1.0133,  ..., -3.0396, -3.5207, -3.5682],
         [-1.1310, -0.7215,  0.0072,  ..

Step,Training Loss,Validation Loss
10,19.449965,2.771398
20,18.685356,2.556762
30,16.850180,2.199665
40,14.390567,1.736558
50,12.341237,1.291568
60,9.904828,0.962445
70,8.118283,0.775801
80,6.957711,0.734839
90,6.114787,0.626948
100,5.888364,0.565092


Writing model shards: 100%|██████████| 1/1 [00:03<00:00,  3.20s/it]


KeyboardInterrupt: 

In [8]:
import torch
import soundfile as sf
from transformers import SpeechT5Processor, SpeechT5ForTextToSpeech, SpeechT5HifiGan

# 1. Load your final saved model
FINAL_MODEL_PATH = "./sinhala_tts_output/final_sinhala_speecht5"
device = "cuda" if torch.cuda.is_available() else "cpu"

processor = SpeechT5Processor.from_pretrained(FINAL_MODEL_PATH)
model = SpeechT5ForTextToSpeech.from_pretrained(FINAL_MODEL_PATH).to(device)
vocoder = SpeechT5HifiGan.from_pretrained("microsoft/speecht5_hifigan").to(device)

# 2. Use a complex sentence to test clarity
test_text = "මෙම ආකෘතිය දැන් ඉතා හොඳින් ක්‍රියා කරන බව පෙනේ." # "This model seems to work very well now."

inputs = processor(text=test_text, return_tensors="pt").to(device)
speaker_embeddings = torch.zeros((1, 512)).to(device) # Use same dummy zeros

# 3. Generate
with torch.no_grad():
    speech = model.generate_speech(inputs["input_ids"], speaker_embeddings, vocoder=vocoder)

# 4. Save and Listen
sf.write("final_test_result.wav", speech.cpu().numpy(), samplerate=16000)
print("Final test audio saved as 'final_test_result.wav'")

Loading weights: 100%|██████████| 158/158 [00:00<00:00, 866.73it/s, Materializing param=upsampler.3.weight]           


Final test audio saved as 'final_test_result.wav'
